In [90]:
original_corpus = [
    "low",
    "new",
    "lowest",
    "newest",
    "widest",
    "lower",
]

In [91]:
def get_vocabulary(corpus: list[str]) -> list[str]:
    return {letter for word in corpus for letter in word} 

In [92]:
def process_corpus(corpus: list[str]) -> list[list[str]]:
    return [[letter for letter in word] for word in corpus]

In [93]:
def count_vocab(corpus: list[list[str]]) -> dict[str, list[int, str, str]]:
    result = {}
    for word in corpus:
        for i in range(len(word) - 1):
            pair = word[i] + word[i+1]
            if pair in result: 
                result[pair][0] += 1
            else:
                result[pair] = [1, word[i], word[i+1]]

    return result

In [94]:
def get_most_freq_pair(counts: dict[str, list[int, str, str]]) -> str:
    max_freq = 0
    max_pair = None
    for key, value in counts.items(): 
        if value[0] > max_freq:
            max_freq = value[0]
            max_pair = (key, value[1], value[2])

    return max_pair

In [95]:
def merge_pair(corpus: list[list[str]], pair: str) -> list[list[str]]:
    result = []
    for word in corpus:
        new_word = []
        i = 0
        while i < len(word) - 1:
            
            if word[i] + word[i+1] == pair:
                new_word.append(pair)
                i+=2
            else:
                new_word.append(word[i])
                i+=1

        if i < len(word):
            new_word.append(word[i])

        result.append(new_word)

    return result

In [96]:
def bpe(corpus: list[str]) -> list[tuple[str, str]]:
    vocab = get_vocabulary(corpus)
    corpus = process_corpus(corpus)
    merges = []

    for i in range(6):
        counts = count_vocab(corpus)
        pair, a, b = get_most_freq_pair(counts)
        vocab.add(pair)
        corpus = merge_pair(corpus, pair)
        merges.append((a, b))

        print(f"Merge {i+1}")
        print(f"Seleceted pair: {a}+{b} = {pair}")
        print(f"New vocab:\n{vocab}")
        print(f"New corpus:\n{corpus}")

    return merges
    

In [97]:
merges = bpe(original_corpus)

Merge 1
Seleceted pair: l+o = lo
New vocab:
{'lo', 'e', 'w', 'o', 's', 'i', 'l', 'd', 't', 'n', 'r'}
New corpus:
[['lo', 'w'], ['n', 'e', 'w'], ['lo', 'w', 'e', 's', 't'], ['n', 'e', 'w', 'e', 's', 't'], ['w', 'i', 'd', 'e', 's', 't'], ['lo', 'w', 'e', 'r']]
Merge 2
Seleceted pair: lo+w = low
New vocab:
{'lo', 'low', 'e', 'w', 'o', 's', 'i', 'l', 'd', 't', 'n', 'r'}
New corpus:
[['low'], ['n', 'e', 'w'], ['low', 'e', 's', 't'], ['n', 'e', 'w', 'e', 's', 't'], ['w', 'i', 'd', 'e', 's', 't'], ['low', 'e', 'r']]
Merge 3
Seleceted pair: e+s = es
New vocab:
{'lo', 'low', 'e', 'w', 'o', 's', 'i', 'l', 'es', 'd', 't', 'n', 'r'}
New corpus:
[['low'], ['n', 'e', 'w'], ['low', 'es', 't'], ['n', 'e', 'w', 'es', 't'], ['w', 'i', 'd', 'es', 't'], ['low', 'e', 'r']]
Merge 4
Seleceted pair: es+t = est
New vocab:
{'lo', 'low', 'e', 'w', 'est', 'o', 's', 'i', 'l', 'es', 'd', 't', 'n', 'r'}
New corpus:
[['low'], ['n', 'e', 'w'], ['low', 'est'], ['n', 'e', 'w', 'est'], ['w', 'i', 'd', 'est'], ['low', 'e'

In [98]:
def test_word(word: str, merges: list[tuple[str, str]]):
    word = [letter for letter in word]
    found = True
    
    while found:
        print(word)
        found = False
        new_word = []
        i = 0
        while i < len(word) - 1:
            if not found and (word[i], word[i+1]) in merges:
                found = True
                new_word.append(word[i] + word[i+1])
                i+=2
            else:
                new_word.append(word[i])
                i+=1

        if i < len(word):
            new_word.append(word[i])

        word = new_word

In [99]:
test_word("slowest", merges)

['s', 'l', 'o', 'w', 'e', 's', 't']
['s', 'lo', 'w', 'e', 's', 't']
['s', 'low', 'e', 's', 't']
['s', 'low', 'es', 't']
['s', 'low', 'est']
